# 🔥 Student Performance Prediction System
### Predict whether a student will **pass/fail** or score **high/low**

**Concepts Covered:**
- 📦 Dataset Handling (Unit III)
- 🤖 Classification (Unit IV)
- 🧹 Data Preprocessing (Lab 6, 7)
- 📊 Visualization (Lab 10)

## ⚙️ Step 1: Install & Import Libraries

In [ ]:
# All libraries come pre-installed in Google Colab!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries imported successfully!')

## 📦 Step 2: Generate / Load Dataset
We'll **synthetically generate** a realistic student dataset so the notebook runs instantly without any CSV file.

In [ ]:
np.random.seed(42)
n = 200

hours_study    = np.random.uniform(0, 10, n)
attendance     = np.random.uniform(50, 100, n)
prev_score     = np.random.uniform(30, 100, n)
assignments    = np.random.randint(0, 11, n)          # out of 10
sleep_hours    = np.random.uniform(4, 10, n)
gender         = np.random.choice(['Male', 'Female'], n)
parental_edu   = np.random.choice(['None', 'High School', 'Bachelor', 'Master'], n)

# Final score depends on above features + noise
final_score = (
    3.5 * hours_study +
    0.3 * attendance +
    0.4 * prev_score +
    1.2 * assignments +
    0.5 * sleep_hours +
    np.random.normal(0, 5, n)
)
final_score = np.clip(final_score, 0, 100)

# Binary label: Pass (1) if score >= 50, else Fail (0)
result = (final_score >= 50).astype(int)

df = pd.DataFrame({
    'Hours_Study'   : np.round(hours_study, 2),
    'Attendance'    : np.round(attendance, 2),
    'Prev_Score'    : np.round(prev_score, 2),
    'Assignments'   : assignments,
    'Sleep_Hours'   : np.round(sleep_hours, 2),
    'Gender'        : gender,
    'Parental_Edu'  : parental_edu,
    'Final_Score'   : np.round(final_score, 2),
    'Result'        : result          # 1 = Pass, 0 = Fail
})

print(f'Dataset shape: {df.shape}')
df.head(10)

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('--- Basic Info ---')
print(df.info())
print('\n--- Summary Statistics ---')
df.describe().round(2)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nPass / Fail distribution:')
print(df['Result'].value_counts().rename({1:'Pass', 0:'Fail'}))

## 📊 Step 4: Visualization

In [ ]:
# --- 4a. Scatter plot with Regression Line (like the slide) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter + Regression
axes[0].scatter(df['Hours_Study'], df['Final_Score'],
                alpha=0.6, color='black', label='Actual Scores')
m, b = np.polyfit(df['Hours_Study'], df['Final_Score'], 1)
x_line = np.linspace(df['Hours_Study'].min(), df['Hours_Study'].max(), 100)
axes[0].plot(x_line, m * x_line + b, color='royalblue', linewidth=2, label='Regression Line')
axes[0].set_title('Student Performance Prediction with Regression Line')
axes[0].set_xlabel('Hours of Study')
axes[0].set_ylabel('Scores')
axes[0].legend()

# Pass/Fail Pie
counts = df['Result'].value_counts()
axes[1].pie(counts, labels=['Pass', 'Fail'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[1].set_title('Pass / Fail Distribution')

# Avg Score by Subject (simulated)
subjects = ['English', 'Math', 'Science', 'Social Studies']
avg_grades = [82.79, 67.41, 75.81, 73.66]
bars = axes[2].bar(subjects, avg_grades,
                   color=['#2196F3','#FF9800','#4CAF50','#9C27B0'])
axes[2].set_title('Average Grade by Subject')
axes[2].set_ylabel('Avg Grade')
axes[2].set_ylim(0, 100)
for bar, val in zip(bars, avg_grades):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- 4b. Feature Correlation Heatmap ---
plt.figure(figsize=(9, 6))
numeric_cols = df.select_dtypes(include=np.number)
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4c. Boxplot: Hours of Study vs Result ---
plt.figure(figsize=(7, 4))
df['Result_Label'] = df['Result'].map({1: 'Pass', 0: 'Fail'})
sns.boxplot(data=df, x='Result_Label', y='Hours_Study',
            palette={'Pass': '#4CAF50', 'Fail': '#F44336'})
plt.title('Hours of Study vs Result')
plt.xlabel('Result')
plt.ylabel('Hours of Study')
plt.tight_layout()
plt.show()

## 🧹 Step 5: Data Preprocessing (Lab 6 & 7)

In [ ]:
# --- Encode categorical variables ---
df_model = df.drop(columns=['Result_Label'])   # drop helper col

le = LabelEncoder()
df_model['Gender']       = le.fit_transform(df_model['Gender'])
df_model['Parental_Edu'] = le.fit_transform(df_model['Parental_Edu'])

# --- Features and Target ---
X = df_model.drop(columns=['Result', 'Final_Score'])   # features
y = df_model['Result']                                  # target (Pass/Fail)

# --- Train/Test Split (80-20) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Feature Scaling ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print('✅ Preprocessing done!')

## 🤖 Step 6: Classification Models (Unit IV)

In [ ]:
# ---- Model 1: Logistic Regression ----
lr = LogisticRegression(random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('=== Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Fail','Pass']))

In [ ]:
# ---- Model 2: Decision Tree ----
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('=== Decision Tree Classifier ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
print(classification_report(y_test, y_pred_dt, target_names=['Fail','Pass']))

In [ ]:
# ---- Confusion Matrices side by side ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_dt],
    ['Logistic Regression', 'Decision Tree']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Fail', 'Pass'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {title}')

plt.tight_layout()
plt.show()

## 📈 Step 7: Model Comparison

In [ ]:
results = {
    'Logistic Regression': accuracy_score(y_test, y_pred_lr),
    'Decision Tree'      : accuracy_score(y_test, y_pred_dt),
}

plt.figure(figsize=(7, 4))
colors = ['#2196F3', '#FF9800']
bars = plt.bar(results.keys(), results.values(), color=colors, width=0.4)
plt.ylim(0, 1.05)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{val:.2%}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔮 Step 8: Predict a New Student

In [ ]:
# Enter a new student's details here ↓
new_student = pd.DataFrame([{
    'Hours_Study' : 6.5,
    'Attendance'  : 85.0,
    'Prev_Score'  : 72.0,
    'Assignments' : 8,
    'Sleep_Hours' : 7.0,
    'Gender'      : 1,    # 0 = Female, 1 = Male
    'Parental_Edu': 2,    # 0=Bachelor,1=HighSchool,2=Master,3=None
}])

new_scaled = scaler.transform(new_student)

pred_lr = lr.predict(new_scaled)[0]
pred_dt = dt.predict(new_student)[0]
label   = {1: '✅ PASS', 0: '❌ FAIL'}

print('======== Prediction for New Student ========')
print(f'  Logistic Regression : {label[pred_lr]}')
print(f'  Decision Tree       : {label[pred_dt]}')

---
## ✅ Summary
| Step | What We Did |
|------|-------------|
| 1 | Imported libraries (NumPy, Pandas, Sklearn, Matplotlib, Seaborn) |
| 2 | Generated a synthetic student dataset (200 rows, 9 features) |
| 3 | Performed EDA — shape, stats, null check, class balance |
| 4 | Visualized — scatter/regression, pie chart, heatmap, boxplot |
| 5 | Preprocessed — encoding, train/test split, feature scaling |
| 6 | Trained **Logistic Regression** and **Decision Tree** classifiers |
| 7 | Compared model accuracies with a bar chart |
| 8 | Made a prediction for a brand-new student |

> **Tip:** Run all cells top-to-bottom with `Runtime → Run All` in Google Colab 🚀